In [ ]:
import triton
import triton.language as tl
import torch

## vector addition

In [ ]:
@triton.jit
def add_kernel(x_ptr,y_ptr,z_ptr,n_elements,BLOCK_SIZE: tl.constexpr):

  pid = tl.program_id(axis=0)
  blockstart = pid * BLOCK_SIZE
  offsets = blockstart + tl.arange(0, BLOCK_SIZE)
  mask = offsets  < n_elements

  x = tl.load(x_ptr + offsets, mask=mask, other=None) # specific part of x vector for this kernel thread
  y = tl.load(y_ptr + offsets, mask=mask, other=None)

  out = x + y
  tl.store(z_ptr + offsets, out, mask=mask)


device = 'cuda'

def add(x: torch.Tensor, y: torch.Tensor):
  z = torch.empty_like(x)
  assert x.device.type == y.device.type == 'cuda'

  n_elements = z.numel()

  grid = lambda meta: (triton.cdiv(n_elements, meta['BLOCK_SIZE']), )      # tuple

  add_kernel[grid](
      x,
      y,
      z,
      n_elements,
      BLOCK_SIZE=1024
  )

  return z




import timeit
x = torch.randn(1000000000, device=device)
y = torch.randn(1000000000, device=device)


start = timeit.default_timer()
z_tri = add(x,y)
end = timeit.default_timer()

end - start

## softmax | operator fusion

In [25]:
import torch
def naive_softmax(x: torch.Tensor):
  return torch.exp(x)/ torch.sum(torch.exp(x))


x = torch.randint(0,10,(3,4))
naive_softmax(x)

tensor([[4.5528e-01, 3.0676e-03, 2.2667e-02, 4.1516e-04],
        [2.2667e-02, 3.0676e-03, 2.2667e-02, 4.5528e-01],
        [4.1516e-04, 3.0676e-03, 3.0676e-03, 8.3387e-03]])

In [26]:
{'max_shared_mem': 65536,
 'max_num_regs': 65536,
 'multiprocessor_count': 40,
 'warpSize': 32,
 'sm_clock_rate': 1590000,
 'mem_clock_rate': 5001000,
 'mem_bus_width': 256}

{'max_shared_mem': 65536,
 'max_num_regs': 65536,
 'multiprocessor_count': 40,
 'warpSize': 32,
 'sm_clock_rate': 1590000,
 'mem_clock_rate': 5001000,
 'mem_bus_width': 256}

In [2]:
import torch
a= torch.tensor([0, 3, 1, 4, 2, 5])

In [4]:
a.is_contiguous()

True

In [22]:
m = torch.tensor([[0,3],[1,4],[2,5]])
m

tensor([[0, 3],
        [1, 4],
        [2, 5]])

In [20]:
aaa = m.view(6)

In [ ]:
aaaaa.view(6)

RuntimeError: view size is not compatible with input tensor's size and stride (at least one dimension spans across two contiguous subspaces). Use .reshape(...) instead.

In [23]:
m = torch.tensor([[0,3],[1,4],[2,5]])
mT = m.T
mT.view(6) # why it doesnt work: RuntimeError: view size is not compatible with input tensor's size and stride (at least one dimension spans across two contiguous subspaces). Use .reshape(...) instead.
m.view(6) # i know it works

RuntimeError: view size is not compatible with input tensor's size and stride (at least one dimension spans across two contiguous subspaces). Use .reshape(...) instead.

In [ ]:
mT.view(6)
mT.contiguous().view(6)

tensor([[0, 1, 2],
        [3, 4, 5]])

In [26]:
mT.contiguous().view(6)

tensor([0, 1, 2, 3, 4, 5])

In [19]:
m.stride()

(2, 1)

In [21]:
aaa.view((2,3))

tensor([[0, 3, 1],
        [4, 2, 5]])

In [52]:
x = torch.randn(2, 1, 3)
x

tensor([[[ 0.1739,  0.0520, -0.4546]],

        [[-0.2593, -1.6507,  0.5228]]])

In [53]:
x.stride()

(3, 3, 1)

In [57]:
xt = x.T

In [58]:
xt.shape

torch.Size([3, 1, 2])

In [59]:
xt

tensor([[[ 0.1739, -0.2593]],

        [[ 0.0520, -1.6507]],

        [[-0.4546,  0.5228]]])

In [ ]:
q = torch.randn((1,6))
q

tensor([[ 0.9119,  0.7731,  0.0037,  0.9438, -0.7842, -0.8907],
        [ 0.3596,  0.9248,  1.1512, -0.3684, -1.0831, -0.2279]])

In [75]:
qt = q.T
qt

tensor([[ 0.9119,  0.3596],
        [ 0.7731,  0.9248],
        [ 0.0037,  1.1512],
        [ 0.9438, -0.3684],
        [-0.7842, -1.0831],
        [-0.8907, -0.2279]])

In [ ]:
qt.view((2,3)) # error

In [ ]:
import torch
class FA2_Forward_PyTorch():
    def __init__(self, B_r=16, B_c=16):
        self.B_r= B_r
        self.B_c= B_c

    def flashattention2_forward_pytorch(self, Q, K, V, is_causal): # Q,K,V - (N,d)
        N,d = Q.shape
        blocksQ = torch.split(Q, self.B_r)
        blocksK = torch.split(K, self.B_c)
        blocksV = torch.split(V, self.B_c)
        O = torch.empty_like(K)
        blocksO = torch.split(O, self.B_r)
        Tr = N // self.B_r
        Tc =  N // self.B_c

        for i in range(Tr):
            Qi = blocksQ[i] #(2,128)
            Oi = blocksO[i]
            # Oi = torch.zeros((self.B_r,d))
            li = torch.zeros(self.B_r)
            mi = torch.ones(self.B_r) * float('-inf')

            for j in range(Tc):
                Kj = blocksK[j] # (3,128)
                Vj = blocksV[j]
                Sij = Qi @ Kj.T
                mij1 = mi
                mi = torch.max(mi, torch.max(Sij, dim=1).values)
                Pij = torch.exp(Sij - mi)
                correction_factor = torch.exp(mij1-mi)
                print(li)
                li = li @ correction_factor + torch.sum(Pij, dim=1)
                Oi = torch.diag(correction_factor) @ Oi  + Pij @ Vj


            blocksO[i] = torch.diag(li) @ Oi

        O = torch.cat(blocksO, dim=-1)

        return O



class FA2Function(torch.autograd.Function):
    B_r = 16
    B_c = 16

    @staticmethod
    def forward(ctx,Q,K,V, is_causal):
        clss = FA2_Forward_PyTorch(FA2Function.B_r, FA2Function.B_c)
        O, Ls = clss.flashattention2_forward_pytorch(Q, K, V, is_causal)

        ctx.save_for_backward(Ls, Q, K, V, O)
        return O

    @staticmethod
    def backward(ctx):
        raise NotImplementedError('not implemented')


In [1]:
clss = FA2_Forward_PyTorch(16,16)
Q, K, V = torch.randn(64,128),  torch.randn(64,128),  torch.randn(64,128)
O, L = clss.flashattention2_forward_pytorch(Q, K, V, is_causal=False)
O.shape

NameError: name 'FA2_Forward_PyTorch' is not defined

### triton - fa2

In [ ]:
import triton
@triton.jit
def fa2_forward_triton(q_ptr, k_ptr, v_ptr, o_ptr,
                       SEQ_lEN:tl.constexpr, HEAD_DIM:tl.constexpr,
                       BLOCK_SIZE_Q:tl.constexpr):

    pid = tl.program_id(axis=0) # is basically which row of Q block matrix
    pass


class FA2_Forward_Triton(torch.autograd.Function):


    @staticmethod
    def forward(self, Q, K, V,is_causal):
        B_q, B_k = 64, 64
        B, N, d = Q.shape
        O = torch.empty_like(Q)
        # BLOCK_SIZE_Q = self.B_r
        grid = lambda meta: triton.cdiv(N, meta['BLOCK_SIZE_Q'])

        fa2_forward_triton[grid](
            Q, K, V, O,
            Q.stride(0), Q.stride(1),
            K.stride(0), K.stride(1),
            V.stride(0), V.stride(1),
            O.stride(0), O.stride(1),
            SEQ_lEN=N, HEAD_DIM=d, BLOCK_SIZE_Q=self.B_r)
        return O

ModuleNotFoundError: No module named 'triton'

In [ ]:
import torch
import math
Q = torch.randn(32,128) # N = 32, br = 2, tr = 16,
blocksQ = torch.split(Q, 2)
blocksQ[15] = torch.zeros


TypeError: 'tuple' object does not support item assignment

In [147]:
a = torch.randn(2)
b = torch.randn(2)
a,b
# torch.max(a )

(tensor([-0.8566,  0.2778]), tensor([-1.7829, -0.7013]))

In [148]:
torch.max(a )

tensor(0.2778)

In [143]:
2**2

4

In [5]:
import torch
t1 = torch.tensor([12,13,14,15, 16,17])
t1

tensor([12, 13, 14, 15, 16, 17])

In [7]:
t1[:, None]

tensor([[12],
        [13],
        [14],
        [15],
        [16],
        [17]])

In [ ]:
get = list(print({"BLOCK_SIZE_Q": BLOCK_SIZE_Q, "BLOCK_SIZE_KV": BLOCK_SIZE_KV})
for BLOCK_SIZE_Q in [64, 128]
for BLOCK_SIZE_KV in [32, 64]
for num_stages in ([3,4,7])
for num_warps in [2,4])

{'BLOCK_SIZE_Q': 64, 'BLOCK_SIZE_KV': 32}
{'BLOCK_SIZE_Q': 64, 'BLOCK_SIZE_KV': 32}
{'BLOCK_SIZE_Q': 64, 'BLOCK_SIZE_KV': 32}
{'BLOCK_SIZE_Q': 64, 'BLOCK_SIZE_KV': 32}
{'BLOCK_SIZE_Q': 64, 'BLOCK_SIZE_KV': 32}
{'BLOCK_SIZE_Q': 64, 'BLOCK_SIZE_KV': 32}
{'BLOCK_SIZE_Q': 64, 'BLOCK_SIZE_KV': 64}
{'BLOCK_SIZE_Q': 64, 'BLOCK_SIZE_KV': 64}
{'BLOCK_SIZE_Q': 64, 'BLOCK_SIZE_KV': 64}
{'BLOCK_SIZE_Q': 64, 'BLOCK_SIZE_KV': 64}
{'BLOCK_SIZE_Q': 64, 'BLOCK_SIZE_KV': 64}
{'BLOCK_SIZE_Q': 64, 'BLOCK_SIZE_KV': 64}
{'BLOCK_SIZE_Q': 128, 'BLOCK_SIZE_KV': 32}
{'BLOCK_SIZE_Q': 128, 'BLOCK_SIZE_KV': 32}
{'BLOCK_SIZE_Q': 128, 'BLOCK_SIZE_KV': 32}
{'BLOCK_SIZE_Q': 128, 'BLOCK_SIZE_KV': 32}
{'BLOCK_SIZE_Q': 128, 'BLOCK_SIZE_KV': 32}
{'BLOCK_SIZE_Q': 128, 'BLOCK_SIZE_KV': 32}
{'BLOCK_SIZE_Q': 128, 'BLOCK_SIZE_KV': 64}
{'BLOCK_SIZE_Q': 128, 'BLOCK_SIZE_KV': 64}
{'BLOCK_SIZE_Q': 128, 'BLOCK_SIZE_KV': 64}
{'BLOCK_SIZE_Q': 128, 'BLOCK_SIZE_KV': 64}
{'BLOCK_SIZE_Q': 128, 'BLOCK_SIZE_KV': 64}
{'BLOCK_SIZE_Q': 128, '

In [17]:
non

<generator object <genexpr> at 0x104bb0940>

In [28]:
import torch
print(torch.randint(0, 10, (3,)))

tensor([2, 4, 8])


In [4]:
import timeit
start= timeit.default_timer()
ia = 0
for i in range(100):
    ia+=i

end = timeit.default_timer()
end-start


5.9667014284059405e-05

In [5]:
ia

4950